# ATTACH, Backup and the sqlite3 CLI

## Quick notes
### What this covers
- `ATTACH DATABASE` to work with several database files (or in-memory databases) on one connection, schema-qualified names (`main.`, `temp.` and the name you give in `ATTACH`), and cross-database joins and copies.
- Transactions that span attached databases, TEMP tables and views, `DETACH` and `PRAGMA database_list`.
- Backups: `VACUUM INTO`, the Python backup API, and `iterdump` (the Python version of `.dump`).
- The `sqlite3` command-line shell (shown as examples, not run here), WAL files, read-only connections and shared in-memory databases.
### Key points
- One connection can have many databases open. They are called `main`, `temp` and whatever name you give in `ATTACH ... AS name`.
- Prefix a table with its database name: `archive.orders`. Without a prefix SQLite searches `temp`, then `main`, then attached databases in the order they were attached.
- One transaction can change several attached files and commit or roll back all of them together.
- `VACUUM INTO 'file'` writes a compact, consistent copy of a live database. It is the easiest backup.
- TEMP tables and views live in the `temp` database. They disappear when the connection closes.
### Gotchas (the part worth revising twice)
- A bare table name can silently pick the wrong database if two databases share a table name. Qualify names when you attach.
- `DETACH` fails ("database is locked") while the current transaction has used that attached database.
- Foreign keys cannot point to a table in another database. Cross-database integrity is your job.
- Atomic commit across files holds in the default rollback-journal mode. In WAL mode, a transaction touching several files is atomic per file but not guaranteed across files after a crash.
- Copying a `.db` file with `cp` while a program writes to it (or while a `-wal` file exists) can give a broken copy. Use `VACUUM INTO` or the backup API.
- `mode=ro` stops writes at the connection level, but a read-only connection still needs the file's `-wal` and `-shm` files to be readable for a WAL database.
- Shared-cache in-memory databases vanish when the last connection to them closes. The shared-cache feature itself is discouraged by SQLite in new code.
- A default build allows 10 attached databases (up to 125 if compiled that way).
### When it matters
- Archiving old rows to another file, merging two databases, scripting data fixes, making nightly backups, and automating tasks with the CLI.
### One-line recap example
- `ATTACH 'archive.db' AS arch; INSERT INTO arch.orders SELECT * FROM main.orders WHERE year = 2022; DETACH arch;` means "copy the 2022 orders into the other file, then let go of it".

## 01. Seed data in the main database
- A small shop: customers and orders. Dates are fixed text.
- Later, old orders are moved to an attached archive database.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT NOT NULL);
INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ben','Leeds'),(3,'Chloe','Oslo');
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date  TEXT NOT NULL,
    total       REAL NOT NULL
);
INSERT INTO orders VALUES
 (1,1,'2022-11-05',40.0),(2,2,'2022-12-19',15.5),(3,1,'2023-01-10',99.0),
 (4,3,'2023-02-02',25.0),(5,2,'2023-02-14',60.0);
SELECT count(*) AS orders_in_main FROM orders;

orders_in_main
--------------
5             
(1 row)



## 02. ATTACH an in-memory database and list databases
- `ATTACH DATABASE ':memory:' AS archive` creates a fresh empty database in memory. A file name works the same way and creates the file if it does not exist.
- `PRAGMA database_list` shows every database on the connection: sequence number, name and file. In-memory databases have an empty file.
- `main` is always seq 0. Seq 1 is reserved for `temp`, which is listed only once something uses it (a TEMP object, see section 08), so `archive` shows as seq 2.

In [3]:
%%sql
ATTACH DATABASE ':memory:' AS archive;
SELECT seq, name, file = '' AS is_in_memory FROM pragma_database_list ORDER BY seq;

seq | name    | is_in_memory
----+---------+-------------
0   | main    | 1           
2   | archive | 1           
(2 rows)



## 03. Schema-qualified names
- Write `schema.table` to say exactly where a table lives: `main.orders`, `archive.orders`, `temp.scratch`.
- Each database has its own catalog: `main.sqlite_schema`, `archive.sqlite_schema`, and so on. (`sqlite_master` is an older name for the same thing.)
- `CREATE TABLE archive.orders (...)` builds the table in the attached database only.

In [4]:
%%sql
CREATE TABLE archive.orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date  TEXT NOT NULL,
    total       REAL NOT NULL
);
SELECT 'main' AS db, name FROM main.sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
UNION ALL
SELECT 'archive', name FROM archive.sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
ORDER BY db, name;

db      | name     
--------+----------
archive | orders   
main    | customers
main    | orders   
(3 rows)



## 04. Copy rows between databases
- `INSERT INTO archive.orders SELECT ... FROM main.orders WHERE ...` copies rows across. It is one statement, so it is atomic.
- Archiving is copy first, then delete from the source. Do both in one transaction (next sections).
- Count both sides to check.

In [5]:
%%sql
INSERT INTO archive.orders SELECT * FROM main.orders WHERE order_date < '2023-01-01';
SELECT 'main' AS db, count(*) AS n FROM main.orders
UNION ALL SELECT 'archive', count(*) FROM archive.orders
ORDER BY db;

db      | n
--------+--
archive | 2
main    | 5
(2 rows)



## 05. Cross-database joins
- You can join tables from different databases in one query, with ordinary join syntax.
- Here the archive has orders but no customers table, so we join archive orders to `main.customers`.
- See 02-Querying/01-Joins for join basics.

In [6]:
%%sql
SELECT c.name, a.order_id, a.order_date, a.total
FROM archive.orders a
JOIN main.customers c ON c.customer_id = a.customer_id
ORDER BY a.order_date, a.order_id;

name | order_id | order_date | total
-----+----------+------------+------
Asha | 1        | 2022-11-05 | 40.0 
Ben  | 2        | 2022-12-19 | 15.5 
(2 rows)



In [7]:
%%sql
-- every order from either database, with where it lives
SELECT 'main' AS db, order_id, order_date FROM main.orders
UNION ALL
SELECT 'archive', order_id, order_date FROM archive.orders
ORDER BY order_date, order_id, db;

db      | order_id | order_date
--------+----------+-----------
archive | 1        | 2022-11-05
main    | 1        | 2022-11-05
archive | 2        | 2022-12-19
main    | 2        | 2022-12-19
main    | 3        | 2023-01-10
main    | 4        | 2023-02-02
main    | 5        | 2023-02-14
(7 rows)



## 06. How an unqualified name is resolved
- SQLite looks in `temp` first, then `main`, then the attached databases in the order they were attached.
- Both `main` and `archive` have `orders`. A bare `orders` means `main.orders`. `archive` is never chosen.
- Lesson: if two databases share table names, always qualify.

In [8]:
%%sql
SELECT 'bare name' AS how, count(*) AS rows_seen FROM orders
UNION ALL SELECT 'main.orders', count(*) FROM main.orders
UNION ALL SELECT 'archive.orders', count(*) FROM archive.orders
ORDER BY how;

how            | rows_seen
---------------+----------
archive.orders | 2        
bare name      | 5        
main.orders    | 5        
(3 rows)



## 07. Archiving in one transaction (atomic across databases)
- One `BEGIN ... COMMIT` can write to several attached databases. Either all changes happen or none.
- Pattern: copy old rows to the archive, delete them from main, commit.
- Below: first a transaction that we roll back, to show that both databases return to their old state, then the real one that commits.

In [9]:
%%sql
BEGIN;
INSERT INTO archive.orders SELECT * FROM main.orders WHERE order_id = 3;
DELETE FROM main.orders WHERE order_id = 3;
SELECT 'inside txn: main' AS where_, count(*) AS n FROM main.orders
UNION ALL SELECT 'inside txn: archive', count(*) FROM archive.orders
ORDER BY where_;
ROLLBACK;
SELECT 'after rollback: main' AS where_, count(*) AS n FROM main.orders
UNION ALL SELECT 'after rollback: archive', count(*) FROM archive.orders
ORDER BY where_;

where_              | n
--------------------+--
inside txn: archive | 3
inside txn: main    | 4
(2 rows)

where_                  | n
------------------------+--
after rollback: archive | 2
after rollback: main    | 5
(2 rows)



In [10]:
%%sql
BEGIN;
INSERT INTO archive.orders SELECT * FROM main.orders WHERE order_date < '2023-02-01' AND order_id NOT IN (SELECT order_id FROM archive.orders);
DELETE FROM main.orders WHERE order_date < '2023-02-01';
COMMIT;
SELECT 'main' AS db, group_concat(order_id) AS order_ids FROM (SELECT order_id FROM main.orders ORDER BY order_id)
UNION ALL SELECT 'archive', group_concat(order_id) FROM (SELECT order_id FROM archive.orders ORDER BY order_id)
ORDER BY db;

db      | order_ids
--------+----------
archive | 1,2,3    
main    | 4,5      
(2 rows)



## 08. TEMP tables and views
- `CREATE TEMP TABLE` (or `TEMP VIEW`) puts the object in the `temp` database. Only this connection sees it, and it is dropped when the connection closes.
- Good for scratch results in a long script, so you do not clutter the real file.
- A TEMP object can shadow a main table of the same name (name resolution checks `temp` first). That is handy for testing and a trap in real code.

In [11]:
%%sql
CREATE TEMP TABLE big_spenders AS
SELECT customer_id, sum(total) AS spent FROM (
    SELECT customer_id, total FROM main.orders
    UNION ALL SELECT customer_id, total FROM archive.orders
) GROUP BY customer_id;
CREATE TEMP VIEW v_big AS SELECT c.name, b.spent FROM big_spenders b JOIN main.customers c USING (customer_id) WHERE b.spent > 50;
SELECT * FROM v_big ORDER BY spent DESC, name;

name | spent
-----+------
Asha | 139.0
Ben  | 75.5 
(2 rows)



In [12]:
%%sql
SELECT seq, name FROM pragma_database_list ORDER BY seq;

seq | name   
----+--------
0   | main   
1   | temp   
2   | archive
(3 rows)



In [13]:
%%sql
SELECT 'temp' AS db, name, type FROM temp.sqlite_schema ORDER BY name;

db   | name         | type 
-----+--------------+------
temp | big_spenders | table
temp | v_big        | view 
(2 rows)



## 09. Shadowing: a TEMP table hides the main table
- A TEMP table named like a main table wins for bare names. Writes without a prefix go to the TEMP one.
- Qualified names still reach the real table.

In [14]:
%%sql
CREATE TEMP TABLE customers (customer_id INTEGER, name TEXT, city TEXT);
INSERT INTO customers VALUES (9, 'Temp Tina', 'Nowhere');
SELECT 'bare name' AS how, name FROM customers
UNION ALL SELECT 'main.customers', name FROM main.customers
ORDER BY how, name;

how            | name     
---------------+----------
bare name      | Temp Tina
main.customers | Asha     
main.customers | Ben      
main.customers | Chloe    
(4 rows)



In [15]:
%%sql
DROP TABLE temp.customers;
SELECT count(*) AS customers_again FROM customers;

customers_again
---------------
3              
(1 row)



## 10. Attaching a real file (tempfile demo)
- A file-backed database behaves the same, but it persists. The demo uses a temporary folder that is deleted at the end of the notebook.
- The path is passed as a parameter so odd characters are safe.
- From here on, `reports` is a file-backed database on the same connection.

In [16]:
import tempfile, pathlib, shutil
_tmp = tempfile.mkdtemp(prefix="sqlite_demo_")
_reports_file = pathlib.Path(_tmp) / "reports.db"
con.execute("ATTACH DATABASE ? AS reports", (str(_reports_file),))
con.execute("CREATE TABLE reports.monthly (month TEXT PRIMARY KEY, revenue REAL NOT NULL)")
con.execute("""INSERT INTO reports.monthly
               SELECT substr(order_date, 1, 7), sum(total)
               FROM (SELECT * FROM main.orders UNION ALL SELECT * FROM archive.orders)
               GROUP BY substr(order_date, 1, 7)""")
print("attached:", [r[1] for r in con.execute("PRAGMA database_list")])

attached: ['main', 'temp', 'archive', 'reports']


In [17]:
%%sql
SELECT month, revenue FROM reports.monthly ORDER BY month;

month   | revenue
--------+--------
2022-11 | 40.0   
2022-12 | 15.5   
2023-01 | 99.0   
2023-02 | 85.0   
(4 rows)



## 11. Constraints stop at the database border
- A foreign key can only point to a table in the same database. You cannot declare one from `reports` to `main.customers`.
- Rows added with a bad `customer_id` in the archive are accepted, because no key exists there.
- If you need the check, do it in a query or a trigger on the same database, or validate before copying.

In [18]:
%%sql
INSERT INTO archive.orders VALUES (900, 4242, '2022-01-01', 1.0);
SELECT a.order_id, a.customer_id, c.customer_id IS NULL AS orphan
FROM archive.orders a LEFT JOIN main.customers c ON c.customer_id = a.customer_id
ORDER BY a.order_id;

order_id | customer_id | orphan
---------+-------------+-------
1        | 1           | 0     
2        | 2           | 0     
3        | 1           | 0     
900      | 4242        | 1     
(4 rows)



In [19]:
%%sql
DELETE FROM archive.orders WHERE order_id = 900;
SELECT count(*) AS archive_rows FROM archive.orders;

archive_rows
------------
3           
(1 row)



## 12. DETACH
- `DETACH DATABASE name` lets go of an attached database. The file is kept; an in-memory one is lost.
- It fails with "database ... is locked" if the open transaction has already used that database (here, an uncommitted insert). Commit or roll back first.
- You cannot detach `main` or `temp`.

In [20]:
%%sql
BEGIN;
INSERT INTO reports.monthly VALUES ('2099-01', 0);

In [21]:
%%sql expect_error
DETACH DATABASE reports;

ERROR: database reports is locked



In [22]:
%%sql
ROLLBACK;
DETACH DATABASE reports;
SELECT seq, name FROM pragma_database_list ORDER BY seq;

seq | name   
----+--------
0   | main   
1   | temp   
2   | archive
(3 rows)



In [23]:
%%sql expect_error
SELECT * FROM reports.monthly;

ERROR: no such table: reports.monthly



## 13. VACUUM INTO: a safe backup
- `VACUUM INTO 'file'` writes a compact copy of the database to a new file. The target must not exist (or must be empty). The copy is a consistent snapshot.
- `VACUUM main INTO 'file'` picks the database. An attached database can be backed up the same way: `VACUUM archive INTO ...`.
- Open the new file in a second connection and count rows to prove it works.

In [24]:
import sqlite3
_backup_file = pathlib.Path(_tmp) / "main_backup.db"
con.execute("VACUUM main INTO ?", (str(_backup_file),))
bk = sqlite3.connect(_backup_file)
print("tables in backup:", [r[0] for r in bk.execute("SELECT name FROM sqlite_schema WHERE type='table' ORDER BY name")])
print("orders in backup:", bk.execute("SELECT count(*) FROM orders").fetchone()[0])
bk.close()

tables in backup: ['customers', 'orders']
orders in backup: 2


## 14. The Python backup API
- `src.backup(dst)` copies a database page by page into another connection. It is safe while others write (it restarts if the source changes) and it can copy to or from memory.
- Use it when you want progress callbacks, or when the target is an existing connection.
- `backup` copies one database (`main` by default), so TEMP tables and other attached databases are not in the copy.
- Below: copy the live database into a fresh in-memory one and query that copy.

In [25]:
mem = sqlite3.connect(":memory:")
con.backup(mem)
print("copy has orders:", mem.execute("SELECT count(*) FROM orders").fetchone()[0])
print("copy has temp tables:", mem.execute("SELECT count(*) FROM sqlite_temp_schema").fetchone()[0])
mem.close()

copy has orders: 2
copy has temp tables: 0


## 15. The sqlite3 command-line shell
- The `sqlite3` program is a shell for SQLite files. It is not run in this notebook; the blocks below are examples to type in a terminal.
- Start it with a file name (created if missing) or with no name for a temporary in-memory database.
- Lines starting with a dot are shell commands (no semicolon). Everything else is SQL and ends with `;`.
- `.open` switches to another file, `.databases` is like `PRAGMA database_list`, `.tables` lists tables (it also takes a LIKE pattern), `.schema` shows the CREATE statements, `.indexes` lists indexes, `.quit` or Ctrl-D leaves.
```text
$ sqlite3 shop.db            # open (or create) a file
$ sqlite3 :memory:           # throwaway in-memory database
$ sqlite3 -readonly shop.db  # open read-only
sqlite> .open other.db
sqlite> .databases
sqlite> .tables
sqlite> .tables 'ord%'
sqlite> .schema orders
sqlite> .indexes orders
sqlite> .quit
```

## 16. Shell: output formatting
- `.mode` chooses how rows print: `list` (default), `column`, `box`, `table`, `markdown`, `csv`, `json`, `insert`, `line`.
- `.headers on` prints column names. `.nullvalue NULL` shows NULL as text instead of blank.
- `.output file` sends results to a file until `.output` is run again with no argument. `.once file` does it for one statement only.
```text
sqlite> .headers on
sqlite> .mode box
sqlite> SELECT name, city FROM customers ORDER BY name;
┌───────┬───────┐
│ name  │ city  │
├───────┼───────┤
│ Asha  │ Pune  │
│ Ben   │ Leeds │
│ Chloe │ Oslo  │
└───────┴───────┘
sqlite> .mode csv
sqlite> .output customers.csv
sqlite> SELECT * FROM customers;
sqlite> .output stdout
```

## 17. Shell: importing data, parameters, timing and plans
- `.import file table` loads a CSV into a table. If the table does not exist, it is created from the header row, with every column as TEXT. Use `--csv` (or `.mode csv` first) so commas are parsed.
- `.parameter set @name value` defines a parameter that SQL can use as `@name`.
- `--skip 1` skips the header row when the table already exists. `.changes on` prints how many rows each statement changed. `.bail on` stops a script at the first error.
- `.timer on` prints the run time of each statement. `.eqp on` shows the query plan before each query. `.explain on` makes `EXPLAIN` output easier to read.
```text
sqlite> .import --csv --skip 1 customers.csv customers
sqlite> .parameter set @city 'Pune'
sqlite> SELECT name FROM customers WHERE city = @city;
sqlite> .timer on
sqlite> .eqp on
sqlite> .changes on
sqlite> .bail on
```

## 18. Shell: scripts, .read, exit codes and -cmd
- `.read script.sql` runs a file of SQL and dot-commands. From the OS you can pipe or redirect: `sqlite3 shop.db < script.sql`.
- `sqlite3 shop.db "SELECT ..."` runs one command and exits. Combine with `-header -csv` for scripting.
- Exit codes: 0 means success. A non-zero code means at least one statement failed (without `-bail` the shell keeps going, but still exits non-zero at the end). Shell scripts can test `$?`.
- A script is just text: statements separated by `;`. Use `BEGIN; ... COMMIT;` inside it for speed and safety.
```text
$ sqlite3 shop.db < migrate.sql
$ sqlite3 -bail shop.db ".read migrate.sql" || echo "migration failed"
$ sqlite3 -header -csv shop.db "SELECT * FROM orders" > orders.csv
$ sqlite3 shop.db "PRAGMA integrity_check"
$ echo $?        # 0 = success
```

## 19. Running a script from Python (the same idea as .read)
- `executescript(text)` runs many statements at once, like `.read`. It does not return rows.
- Reading the file with `pathlib.Path.read_text()` and passing it in gives the `sqlite3 db < script.sql` behaviour.
- The temporary folder keeps the demo clean.

In [26]:
_script = pathlib.Path(_tmp) / "migrate.sql"
_script.write_text("""
BEGIN;
CREATE TABLE IF NOT EXISTS audit_log (id INTEGER PRIMARY KEY, note TEXT NOT NULL);
INSERT INTO audit_log (note) VALUES ('migration 001 applied'), ('migration 002 applied');
COMMIT;
""")
con.executescript(_script.read_text())
print(con.execute("SELECT id, note FROM audit_log ORDER BY id").fetchall())

[(1, 'migration 001 applied'), (2, 'migration 002 applied')]


## 20. A .dump look-alike: iterdump
- The shell command `.dump` prints SQL text that recreates a database (schema and data). Python's `iterdump()` does the same.
- Save the text, and later reload with `executescript`, or in the shell with `sqlite3 new.db < dump.sql`. Dumps are great for version control and for moving data between SQLite versions.
- Below: dump a small database, show the text, then reload it into a brand new one.
```text
$ sqlite3 shop.db .dump > shop.sql
$ sqlite3 restored.db < shop.sql
```

In [27]:
src = sqlite3.connect(":memory:")
src.executescript("""
CREATE TABLE pets (pet_id INTEGER PRIMARY KEY, name TEXT NOT NULL, species TEXT);
INSERT INTO pets VALUES (1,'Rex','dog'),(2,'Tom','cat'),(3,'Nemo',NULL);
CREATE INDEX idx_pets_species ON pets(species);
""")
dump_text = "\n".join(src.iterdump())
print(dump_text)

BEGIN TRANSACTION;
CREATE TABLE pets (pet_id INTEGER PRIMARY KEY, name TEXT NOT NULL, species TEXT);
INSERT INTO "pets" VALUES(1,'Rex','dog');
INSERT INTO "pets" VALUES(2,'Tom','cat');
INSERT INTO "pets" VALUES(3,'Nemo',NULL);
CREATE INDEX idx_pets_species ON pets(species);
COMMIT;


In [28]:
restored = sqlite3.connect(":memory:")
restored.executescript(dump_text)
print(restored.execute("SELECT * FROM pets ORDER BY pet_id").fetchall())
print([r[0] for r in restored.execute("SELECT name FROM sqlite_schema WHERE type = 'index'")])
src.close(); restored.close()

[(1, 'Rex', 'dog'), (2, 'Tom', 'cat'), (3, 'Nemo', None)]
['idx_pets_species']


## 21. WAL mode and its extra files
- The default journal mode is a rollback journal: a temporary `-journal` file is used while writing. In WAL mode (`PRAGMA journal_mode = WAL`) writes go to a `-wal` file, and a `-shm` file coordinates readers.
- After the last connection closes cleanly, SQLite checkpoints and deletes the `-wal` and `-shm` files (the last line of the demo shows this).
- WAL lets readers and one writer work at the same time. The `-wal` content is merged into the main file by a checkpoint (automatic, or `PRAGMA wal_checkpoint`).
- When you copy or move a WAL database file, take the `-wal` and `-shm` files too, or better, use `VACUUM INTO` or the backup API.
- WAL needs a real file on a local disk, so the demo uses a temporary file. Only the file name endings are printed.

In [29]:
_wal_file = pathlib.Path(_tmp) / "wal_demo.db"
w = sqlite3.connect(_wal_file, isolation_level=None)
print("journal_mode:", w.execute("PRAGMA journal_mode = WAL").fetchone()[0])
w.execute("CREATE TABLE t (x INTEGER)")
w.execute("INSERT INTO t VALUES (1), (2), (3)")
print("files present:", sorted(p.name.replace("wal_demo.db", "<db>") for p in pathlib.Path(_tmp).glob("wal_demo.db*")))

journal_mode: wal
files present: ['<db>', '<db>-shm', '<db>-wal']


In [30]:
busy, log_frames, checkpointed = w.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
print("checkpoint busy flag:", busy)
w.close()
print("files after close:", sorted(p.name.replace("wal_demo.db", "<db>") for p in pathlib.Path(_tmp).glob("wal_demo.db*")))

checkpoint busy flag: 0
files after close: ['<db>']


## 22. Read-only connections with URIs
- A URI connection (`file:path?mode=ro`) opens the file read-only. Pass `uri=True` to Python's `connect`.
- Any write gives "attempt to write a readonly database". Great for reporting jobs that must never change data.
- `immutable=1` goes further and tells SQLite the file cannot change (no locking). Only use it on truly read-only media.

In [31]:
_ro_uri = pathlib.Path(_wal_file).as_uri() + "?mode=ro"
ro = sqlite3.connect(_ro_uri, uri=True)
print("rows readable:", ro.execute("SELECT count(*) FROM t").fetchone()[0])
try:
    ro.execute("INSERT INTO t VALUES (4)")
except sqlite3.OperationalError as exc:
    print("write blocked:", exc)
ro.close()

rows readable: 3
write blocked: attempt to write a readonly database


## 23. Named in-memory databases with shared cache
- Plain `:memory:` gives every connection its own private database. A URI like `file:name?mode=memory&cache=shared` gives a named in-memory database that several connections in the same process share.
- It lives until the last connection closes. Useful in tests and for several threads working on one in-memory database.
- SQLite recommends against shared-cache mode in new designs because it has its own locking quirks; the alternative is a real file or WAL.

In [32]:
a = sqlite3.connect("file:demo_shared?mode=memory&cache=shared", uri=True)
b = sqlite3.connect("file:demo_shared?mode=memory&cache=shared", uri=True)
a.execute("CREATE TABLE shared_t (x INTEGER)"); a.execute("INSERT INTO shared_t VALUES (1), (2)"); a.commit()
print("b sees:", b.execute("SELECT count(*) FROM shared_t").fetchone()[0])
private1 = sqlite3.connect(":memory:"); private2 = sqlite3.connect(":memory:")
private1.execute("CREATE TABLE p (x)")
try:
    private2.execute("SELECT * FROM p")
except sqlite3.OperationalError as exc:
    print("private memory dbs are separate:", exc)
for c in (a, b, private1, private2): c.close()

b sees: 2
private memory dbs are separate: no such table: p


## 24. Cleanup
- Detach what is attached and delete the temporary folder.
- Always clean up demo files: a forgotten temp folder is just clutter, but a forgotten attached file can stay locked.

In [33]:
con.execute("DETACH DATABASE archive")
shutil.rmtree(_tmp, ignore_errors=True)
print("attached now:", [r[1] for r in con.execute("PRAGMA database_list")])
print("temp folder removed:", not pathlib.Path(_tmp).exists())

attached now: ['main', 'temp']
temp folder removed: True
